## Question 1 — environment check

Run `uv --version` and record the version in your notes. This is a setup check, not a graded answer: your local `uv` version is not a property of the model.

In [1]:
!uv --version

uv 0.13.6 (x86_64-unknown-linux-gnu)


Answer: 0.13.6

## Question 2 — locked dependency

Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the 2026 reference environment?
- 1.6.1
- 1.7.2
- 1.8.0
- 2.0.0

Do not regenerate the lockfile for the graded run; use `uv sync --locked`.

In [2]:
!grep -A2 'name = "scikit-learn"' uv.lock | grep version

version = "1.7.2"


In [3]:
!uv run python -c "import sklearn; print(sklearn.__version__)"

1.7.2


Answer: 1.7.2

## Question 3 — load the model

Write a Python script that loads `pipeline.bin` with `pickle` and calls `predict_proba` for this lead:

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

Report the probability of conversion rounded to three decimal places. The grader accepts an absolute error of at most 0.005; do not choose the nearest answer from a list.

In [5]:
import pickle

with open('pipeline.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)

pipeline

,steps,"[('dictvectorizer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,dtype,<class 'numpy.float64'>
,separator,'='
,sparse,True
,sort,True
,penalty,'l2'
,dual,False
,tol,0.0001


In [9]:
import sklearn

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

probability = pipeline.predict_proba([lead_q3])[0, 1]
probability, round(probability, 3)

(np.float64(0.5329944186432507), np.float64(0.533))

Answer: 0.533

## Question 4 — serve the model

Start the provided API:
```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

The API has `GET /health` and `POST /predict`. Send this second lead:
```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

For example:
```bash
curl -s http://localhost:9696/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
  }'
```

Report `conversion_probability` rounded to three decimal places. The same `0.005` absolute tolerance applies. The response must contain a probability in `[0, 1]` and a boolean `conversion` field.

In [16]:
%%bash
URL="http://localhost:9696/predict"
DATA='{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}'

RESPONSE=$(curl -s -X POST "$URL" \
  -H 'Content-Type: application/json' \
  -d "$DATA")

echo "Raw response: $RESPONSE"

ROUNDED_PROB=$(echo "$RESPONSE" | jq '.conversion_probability | (.*100 | round / 100)')
echo "Rounded probability: $ROUNDED_PROB"

Raw response: {"conversion_probability":0.769799,"conversion":true}
Rounded probability: 0.77


## Container

The checked-in `Dockerfile` is the canonical 2026 container configuration. It uses Python `3.11.15-slim-bookworm`, `uv 0.10.11`, and the checked-in `uv.lock`. It copies the frozen artifact and starts the API on port `9696`.

Build and run it:
```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

## Question 5 — look at the container configuration

Which Python base-image tag is declared in the Dockerfile?
- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

This question checks a version-controlled declaration. The displayed size of a local Docker image is not graded because it varies by platform, architecture, Docker version, and cached layers.

In [17]:
!head -n 1 Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


Answer: `python:3.11.15-slim-bookworm`

## Question 6 — run the container

Run the same request from Question 4 against the container. You can use the provided smoke client:
```bash
python q6_test.py
```

Report `conversion_probability` rounded to three decimal places. The expected result is the same reference inference as Question 4, with the same `0.005` absolute tolerance. Also verify that:
```bash
curl -s http://localhost:9696/health
```

returns `{"status":"ok", ...}` and the model checksum recorded in `model_metadata.json`.

In [27]:
from __future__ import annotations

import requests


url = "http://localhost:9696/predict"
client = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(url, json=client)
result = response.json()

assert set(result) == {"conversion_probability", "conversion"}
assert 0 <= result["conversion_probability"] <= 1
assert isinstance(result["conversion"], bool)
print(result)
print(f'Rounded probability: {round(result["conversion_probability"], 3)}')

{'conversion_probability': 0.769799, 'conversion': True}
Rounded probability: 0.77


In [28]:
!curl -s http://localhost:9696/health

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}